# 18 — Agent and Multi-Agent Prompt Contracts

    ## Scenario and success criteria

    A support router may look up an order or summarize evidence, but only inside the authenticated tenant and a bounded step budget.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Validate typed tasks at the boundary.
- Authorize before capability exposure.
- Compare a single workflow with a multi-agent design.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 18 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

A model-generated tenant, approval, or role is untrusted and cannot authorize work.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab18 import AgentTask, architecture_cost, route
from northstar.security import Principal

owner = Principal(user_id="u1", tenant="northstar", roles={"support_agent"})
intruder = Principal(user_id="u2", tenant="other", roles={"support_agent"})
task = AgentTask(task_id="T1", tenant="northstar", kind="lookup", payload={"order_id": "O1"}, max_steps=2)

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
print("owner", route(task, owner))
print("cross-tenant", route(task, intruder))
print("single workflow cost", architecture_cost(agents=1, model_calls=2, handoffs=0))
print("three-agent cost", architecture_cost(agents=3, model_calls=4, handoffs=2))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert route(task, owner).status == "completed"
assert route(task, intruder).reason_code == "tenant_mismatch"
assert architecture_cost(agents=1, model_calls=2, handoffs=0) < architecture_cost(agents=3, model_calls=4, handoffs=2)

## Production upgrade

Use narrow tools, explicit terminal states, idempotency keys, bounded retries, durable state only where needed, and approval before consequential side effects. Add agents only when measured decomposition benefits exceed coordination cost.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.